<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 一书的补充代码，作者为 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 内存高效的模型权重加载

- 本 notebook 提供了一些在 GPU（或 CPU）内存有限时加载更大预训练或微调模型的技巧
- 具体来说，它专注于以下情况：你使用 `torch.save(model.state_dict(), "model.pth")` 保存了模型（例如在第 5-7 章中），并希望稍后在新会话中加载它以进行继续预训练或额外微调
- 虽然该示例使用 LLM，但本 notebook 中解释的方法是通用的，适用于加载任何 PyTorch 模型，而不仅仅是 LLM

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/memory-efficient-loading/memory-efficient-loading.webp" width="800px">

In [1]:
from importlib.metadata import version

pkgs = [
    "torch",
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

torch version: 2.9.1+cu130


&nbsp;
## 1. Benchmark 工具

- 首先，让我们定义一些工具代码来跟踪 VRAM（GPU 内存）
- 稍后，我们还将引入一个工具来跟踪主系统 RAM（CPU 内存）
- 当我们稍后应用这些函数时，它们的作用将变得清晰

In [2]:
import gc
import time
import torch


def start_memory_tracking():
    """Initialize GPU memory tracking."""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    else:
        print("This notebook is intended for CUDA GPUs but CUDA is not available.")

def print_memory_usage():
    max_gpu_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)  # Convert bytes to GB
    print(f"Maximum GPU memory allocated: {max_gpu_memory:.1f} GB")

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(3)  # some buffer time to allow memory to clear
    torch.cuda.reset_peak_memory_stats()
    max_memory_allocated = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
    print(f"Maximum GPU memory allocated: {max_memory_allocated:.1f} GB")

&nbsp;
## 2. 模型设置

- 此代码部分设置模型本身
- 在这里，我们使用 "large" GPT-2 模型以增加趣味性（你可以使用 "gpt2-small (124M)" 来降低本 notebook 的内存需求和执行时间）

In [3]:
from previous_chapters import GPTModel
# If the `previous_chapters.py` file is not available locally,
# you can import it from the `llms-from-scratch` PyPI package.
# For details, see: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# E.g.,
# from llms_from_scratch.ch04 import GPTModel



BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-xl (1558M)"

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

- 现在，让我们看看 GPU 内存函数的实际效果：

In [4]:
start_memory_tracking()


model = GPTModel(BASE_CONFIG)
device = torch.device("cuda")
model.to(device)

print_memory_usage()

/home/rasbt/jupyterlab/reasoning/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(


Maximum GPU memory allocated: 6.4 GB


- 此外，让我们通过传入一些示例张量来确保模型正常运行

In [5]:
# Test if the model works (no need to track memory here)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

- 接下来，假设我们正在预训练模型并保存它以供以后使用
- 为简单起见，我们在此跳过实际的预训练，仅保存初始化的模型（但同样的概念适用）

In [6]:
# Training code would go here...

model.train()
torch.save(model.state_dict(), "model.pth")

- 最后，我们在 Python 会话中删除模型和示例张量以重置 GPU 内存

In [7]:
del model, test_input
cleanup()

Maximum GPU memory allocated: 0.0 GB


&nbsp;
## 3. 基本权重加载

- 现在开始有趣的部分，我们加载预训练模型权重
- 让我们看看加载先前保存的模型需要多少 GPU 内存

In [8]:
# Then load pretrained weights

start_memory_tracking()

model = GPTModel(BASE_CONFIG)
model.to(device)

model.load_state_dict(
    torch.load("model.pth", map_location=device, weights_only=True)
)
model.to(device)
model.eval();

print_memory_usage()

Maximum GPU memory allocated: 12.8 GB


- 请注意，内存是上一会话中的两倍
- 这是因为我们在短时间内内存中有相同的模型两次：
  - 第一次是通过 `model.to(device)`
  - 第二次是通过代码行 `model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))`；最终，加载的模型权重将被复制到模型中，并且 `state_dict` 将被丢弃，但在很短的时间内，我们的主模型和加载的 `state_dict` 都在内存中
- 接下来的部分集中解决这个问题
- 但首先，让我们测试模型并重置 GPU 内存

In [9]:
# Test if the model works (no need to track memory here)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

del model, test_input
cleanup()

Maximum GPU memory allocated: 0.0 GB


- 让我们测试另一种在实践中非常流行的常见模式：

In [10]:
start_memory_tracking()

model = GPTModel(BASE_CONFIG)

model.load_state_dict(
    torch.load("model.pth", map_location="cpu", weights_only=True)
)
model.to(device)
model.eval();

print_memory_usage()

Maximum GPU memory allocated: 6.4 GB


In [11]:
# Test if the model works (no need to track memory here)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

del model, test_input
cleanup()

Maximum GPU memory allocated: 0.0 GB


- 因此，就峰值内存而言，我们先在设备上实例化模型然后使用 `map_location="device"`，还是先加载权重到 CPU 内存（`map_location="cpu"`）然后再将其移动到设备上，这两者之间没有区别

&nbsp;
## 4. 按顺序加载权重

- 对于上一节中突出显示的模型权重在 GPU 内存中出现两次的问题，一种变通方法是按顺序加载模型
- 下面，我们：
  - 首先将模型加载到 GPU 内存
  - 然后将模型权重加载到 CPU 内存
  - 最后将每个参数逐个复制到 GPU 内存

In [ ]:
start_memory_tracking()

model = GPTModel(BASE_CONFIG).to(device)

state_dict = torch.load("model.pth", map_location="cpu", weights_only=True)

print_memory_usage()

# Sequentially copy weights to the model's parameters
with torch.no_grad():
    for name, param in model.named_parameters():
        if name in state_dict:
            param.copy_(state_dict[name].to(device))
        else:
            print(f"Warning: {name} not found in state_dict.")

print_memory_usage()

Maximum GPU memory allocated: 6.4 GB
Maximum GPU memory allocated: 6.7 GB


- 正如我们上面所看到的，内存使用量比以前低得多
- 请注意，内存从 6.4 GB 增加到 6.7 GB，因为最初我们只有模型在内存中，然后我们有模型加上 1 个参数张量在内存中（我们临时将参数张量移动到 GPU，以便我们可以使用 `".to"` 赋值给模型）
- 总的来说，这是一个显著的改进
- 同样，让我们简要测试模型，然后重置 GPU 内存以用于下一节

In [ ]:
# Test if the model works (no need to track memory here)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

del model, test_input, state_dict, param
cleanup()

Maximum GPU memory allocated: 0.0 GB


&nbsp;
## 5. 使用低 CPU 内存加载模型

- 在上一会话中，我们通过先将权重（`state_dict`）加载到 CPU 内存，然后逐个将它们复制到模型中来减少 GPU 内存使用
- 但是，如果我们的 CPU 内存有限怎么办？
- 本节使用 PyTorch 所谓的 `"meta"` device 方法在 GPU 内存大但 CPU 内存小的机器上加载模型
- 但首先，让我们定义一个监控 CPU 内存的便捷函数

In [ ]:
import os
import psutil
from threading import Thread


def memory_usage_in_gb(func, *args, **kwargs):
    process = psutil.Process(os.getpid())

    # Measure the baseline memory usage before running the function
    baseline_mem = process.memory_info().rss / 1024 ** 3  # in GB

    # Start monitoring memory in a separate thread
    mem_usage = []
    done = False

    def monitor_memory():
        while not done:
            mem_usage.append(process.memory_info().rss / 1024 ** 3)  # Convert to GB
            time.sleep(0.1)

    t = Thread(target=monitor_memory)
    t.start()

    # Run the function
    func(*args, **kwargs)

    # Stop monitoring
    done = True
    t.join()

    peak_mem_usage_gb = max(mem_usage) - baseline_mem
    return peak_mem_usage_gb


- 首先，让我们跟踪上一节中顺序权重加载方法的 CPU 内存

In [ ]:
def load_sequentially():
    start_memory_tracking()

    model = GPTModel(BASE_CONFIG).to(device)

    state_dict = torch.load("model.pth", map_location="cpu", weights_only=True)

    print_memory_usage()

    # Sequentially copy weights to the model's parameters
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in state_dict:
                param.copy_(state_dict[name].to(device))
            else:
                print(f"Warning: {name} not found in state_dict.")

    print_memory_usage()


peak_memory_used = memory_usage_in_gb(load_sequentially)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 6.4 GB
Maximum GPU memory allocated: 6.7 GB
-> Maximum CPU memory allocated: 6.3 GB


- 现在，假设我们有一台 CPU 内存低但 GPU 内存大的机器
- 我们可以通过引入 PyTorch 所谓的 "meta" device 来权衡 CPU 内存和 GPU 内存的使用
- PyTorch 的 meta device 是一种特殊的设备类型，允许你创建张量而不为其数据分配实际内存，从而有效地创建 "meta" 张量
- 这对于模型分析或架构定义等任务很有用，在这些任务中，你需要张量形状和类型而无需承担内存分配的开销

In [ ]:
def load_sequentially_with_meta():
    start_memory_tracking()

    with torch.device("meta"):
        model = GPTModel(BASE_CONFIG)

    model = model.to_empty(device=device)

    state_dict = torch.load("model.pth", map_location=device, weights_only=True)

    print_memory_usage()

    # Sequentially copy weights to the model's parameters
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in state_dict:
                param.copy_(state_dict[name])
            else:
                print(f"Warning: {name} not found in state_dict.")

    print_memory_usage()

peak_memory_used = memory_usage_in_gb(load_sequentially_with_meta)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 12.8 GB
Maximum GPU memory allocated: 12.8 GB
-> Maximum CPU memory allocated: 1.3 GB


- 正如我们上面所看到的，通过在 meta-device 上创建模型并将权重直接加载到 GPU 内存，我们有效地减少了 CPU 内存需求
- 有人可能会问："那么，顺序权重加载仍然是必要的吗？这与原始方法相比如何？"
- 让我们检查一下用于比较的简单 PyTorch 权重加载方法（来自本 notebook 中的第一个权重加载部分）：

In [ ]:
def baseline():
    start_memory_tracking()

    model = GPTModel(BASE_CONFIG)
    model.to(device)

    model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))
    model.to(device)
    model.eval();

    print_memory_usage()

peak_memory_used = memory_usage_in_gb(baseline)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 12.8 GB
-> Maximum CPU memory allocated: 4.4 GB


- 正如我们上面所看到的，没有 meta device 的"简单"权重加载使用了更多内存
- 换句话说，如果你有一台 CPU 内存有限的机器，你可以使用 meta device 方法将模型权重直接加载到 GPU 内存中，以减少峰值 CPU 内存使用

&nbsp;
## 6. 使用 `mmap=True`（推荐）

- 作为中级或高级 `torch.load` 用户，你可能想知道这些方法与 PyTorch 中的 `mmap=True` 设置相比如何
- PyTorch 中的 `mmap=True` 设置启用 memory-mapped 文件 I/O，它允许张量直接从磁盘存储访问数据，从而通过在 RAM 受限时不将整个文件加载到 RAM 中来减少内存使用
- 另请参阅 [mikaylagawarecki](https://github.com/rasbt/LLMs-from-scratch/issues/402) 的有用评论
- 乍一看，它可能看起来不如上面的顺序方法高效：

In [ ]:
def best_practices():
  with torch.device("meta"):
      model = GPTModel(BASE_CONFIG)

  model.load_state_dict(
      torch.load("model.pth", map_location=device, weights_only=True, mmap=True),
      assign=True
  )

  print_memory_usage()

peak_memory_used = memory_usage_in_gb(best_practices)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 6.4 GB
-> Maximum CPU memory allocated: 5.9 GB


- CPU RAM 使用率如此之高的原因是这台机器上有足够的 CPU RAM 可用
- 但是，如果你在 CPU RAM 有限的机器上运行它，`mmap` 方法将使用更少的内存

&nbsp;
## 7. 其他方法

- 本 notebook 专注于在 PyTorch 中加载权重的简单、内置方法
- 对于 CPU 内存有限的情况，推荐的方法是上面解释的 `mmap=True` 方法
- 或者，另一种选择是一种暴力方法，分别保存和加载每个权重张量：

In [ ]:
model = GPTModel(BASE_CONFIG)
# Assume `model` is your trained model
state_dict = model.state_dict()

# Create a directory to store individual parameter files
os.makedirs("model_parameters", exist_ok=True)

# Save each parameter tensor separately
for name, param in state_dict.items():
    torch.save(param.cpu(), f"model_parameters/{name}.pt")

del model

In [ ]:
def load_individual_weights():

    start_memory_tracking()

    with torch.device("meta"):
        model = GPTModel(BASE_CONFIG)

    model = model.to_empty(device=device)

    print_memory_usage()
    param_dir = "model_parameters"

    with torch.no_grad():
        for name, param in model.named_parameters():
            weight_path = os.path.join(param_dir, f"{name}.pt")
            if os.path.exists(weight_path):
                param_data = torch.load(weight_path, map_location="cpu", weights_only=True)
                param.copy_(param_data)
                del param_data  # Free memory
            else:
                print(f"Warning: {name} not found in {param_dir}.")

    print_memory_usage()


peak_memory_used = memory_usage_in_gb(load_individual_weights)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 6.4 GB
Maximum GPU memory allocated: 6.4 GB
-> Maximum CPU memory allocated: 0.3 GB
